# TD(0) Value Prediction with Streaming iPC — Sweep Analysis

## Background

The Rotating MNIST experiments showed streaming iPC works mechanically, but couldn't distinguish genuine learning from inertia (label constant for 72 steps, trivial baseline at 98.61%). This experiment tests whether streaming iPC's persistent value nodes provide a genuine warm-start advantage in a setting where predictions must change every step.

## Task

**Continuous Grid World Value Prediction**: An agent performs a random walk in [0,1]² with Gaussian steps (σ controls step size), reflected at rectangular barriers and domain boundaries. The agent receives reward +1 inside two circular reward regions. A PC network maps the 2D position to a scalar value prediction V̂(s), trained with TD(0) bootstrap targets.

## Core Hypothesis

Persistent value nodes provide a warm-start advantage: when consecutive states are similar, the representation from s_t is a useful initialization for s_{t+1}. This advantage should:
1. **Scale with temporal correlation** (σ experiment): smaller σ → stronger advantage
2. **Be specifically absent at barrier crossings** (stratification): warm starts are misleading when crossing a barrier
3. **Compensate for limited inference** (T experiment): streaming at T=1 should match forward-init at higher T

## Methods

- **Streaming iPC**: Value nodes persist across observations
- **Forward-init iPC**: Value nodes reinitialized via top-down forward pass each step
- **BP (Adam)**: Standard MLP trained with semi-gradient TD(0) + Adam optimizer
- **Linear BP**: Linear model baseline (demonstrates nonlinearity demand)

In [ ]:
# Data collection command:
# python -m phd.research_utils.scripts.mlflow_download \
#     --tracking-uri=$MLFLOW_TRACKING_URI \
#     --experiment=td-value-prediction \
#     --output-dir=data/

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import TwoSlopeNorm
import seaborn as sns
from IPython.display import display

from phd.research_utils.analysis.analysis_utils import *
from phd.research_utils.analysis.plotting import plot_learning_curves, plot_param_sensitivity
from phd.sandbox.predictive_coding.environment import GridWorld, DEFAULT_BARRIERS, DEFAULT_REWARDS
from phd.sandbox.predictive_coding.value_function import compute_true_value_function

sns.set_theme('notebook', 'white')
%matplotlib inline

save_path = 'figures/'
config_data_path = 'data/td_value_prediction_params.csv'
run_data_path = 'data/td_value_prediction_metrics.csv'

config_dfs, run_dfs = load_experiment_data(config_data_path, run_data_path)
{k: len(v) for k, v in config_dfs.items()}

## Figure 1: True Value Function and Environment

Visualize the task: true V* as a heatmap with barriers and reward regions overlaid. This establishes what the network needs to learn — smooth gradients in open areas, sharp discontinuities at barriers.

In [ ]:
def plot_barrier_overlay(ax):
    """Add barrier rectangles and reward circles to an axis."""
    for bx, by, bw, bh in DEFAULT_BARRIERS:
        ax.add_patch(plt.Rectangle((by, bx), bh, bw, fill=True, color='black', alpha=0.8))
    for r in DEFAULT_REWARDS:
        cx, cy = r['center']
        circle = plt.Circle((cy, cx), r['radius'], fill=False, color='red', linewidth=2)
        ax.add_patch(circle)

# Compute V* at default sigma
gw = GridWorld(sigma=0.03)
v_star_grid, v_star_interp, xs, ys = compute_true_value_function(gw, resolution=100, n_samples=500, seed=0)

fig, ax = plt.subplots(1, 1, figsize=(7, 6))
# Note: v_star_grid[i,j] has i=x index, j=y index. imshow expects (rows=y, cols=x), so transpose.
im = ax.imshow(v_star_grid.T, origin='lower', extent=[0, 1, 0, 1], cmap='viridis', aspect='equal')
plot_barrier_overlay(ax)
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('True Value Function V*(s), γ=0.95, σ=0.03')
plt.colorbar(im, ax=ax, label='V*(s)')
plt.tight_layout()
plt.savefig(save_path + 'fig1_true_value_function.png', dpi=150, bbox_inches='tight')
plt.show()

## Figure 2: LR Sensitivity per Variant

Find the stable operating range for each method before comparing them. The Rotating MNIST experiments showed streaming iPC needs much smaller α than forward-init (64x difference), so we sweep a wide range.

In [ ]:
# LR sensitivity for iPC variants
sweep_label_map = {
    'td_streaming_ipc_lr': 'Streaming iPC',
    'td_forward_init_ipc_lr': 'Forward Init iPC',
    'td_bp_adam_lr': 'BP (Adam)',
    'td_linear_bp_lr': 'Linear BP',
}

# Compute baselines for reference
eval_pts, _ = gw.get_eval_grid(50)
true_vals = v_star_interp(eval_pts)
msve_predict_zero = float(np.mean(true_vals**2))
msve_predict_mean = float(np.mean((true_vals - true_vals.mean())**2))
print(f'Baselines: MSVE(predict 0) = {msve_predict_zero:.3f}, MSVE(predict mean) = {msve_predict_mean:.3f}')

for sweep_name, label in sweep_label_map.items():
    if sweep_name not in config_dfs:
        print(f'  Skipping {sweep_name} (not found)')
        continue
    x_col = 'ipc.alpha' if 'ipc' in sweep_name else 'optimizer.learning_rate'
    plot_param_sensitivity(
        run_df=run_dfs[sweep_name],
        config_df=config_dfs[sweep_name],
        x_col=x_col,
        title=f'{label} — LR Sensitivity',
        x_label='Learning Rate',
        y_label='MSVE',
        metric_col='msve',
        metric_type='final_avg',
        pow_2_x_axis=True,
    )
    plt.axhline(msve_predict_mean, color='grey', linestyle='--', linewidth=1, alpha=0.7, label='Predict mean')
    plt.legend()
    plt.savefig(save_path + f'fig2_lr_{sweep_name}.png', dpi=150, bbox_inches='tight')
    plt.show()

## Figure 3: Variant Comparison at Best LR

Compare all methods at their individually tuned learning rates. MSVE learning curves show convergence speed and final quality.

In [ ]:
# Best LR per variant
variant_sweeps = [k for k in sweep_label_map if k in config_dfs]
variant_config_dfs = {k: config_dfs[k] for k in variant_sweeps}
variant_run_dfs = {k: run_dfs[k] for k in variant_sweeps}

best_config_dfs, best_run_dfs, best_vals = get_best_ablation_values(
    variant_config_dfs, variant_run_dfs,
    metric_col='msve',
    metric_direction='min',
    metric_type='final_avg',
)

print("Best LR per variant:")
for k, v in best_vals.items():
    print(f"  {sweep_label_map.get(k, k)}: {v}")

# Combine for comparison plot
combined_run_df = pd.concat(list(best_run_dfs.values()))
combined_config_df = pd.concat(
    [df.assign(variant=k) for k, df in best_config_dfs.items()]
)
combined_config_df['Algorithm'] = combined_config_df['variant'].map(sweep_label_map)
valid_ids = combined_config_df['run_id'].unique()
combined_run_df = combined_run_df[combined_run_df['run_id'].isin(valid_ids)].copy()

plot_learning_curves(
    run_df=combined_run_df,
    config_df=combined_config_df,
    n_bins=200,
    figsize=(10, 5),
    y_col='msve',
    y_label='MSVE',
    hue_col='Algorithm',
    legend_title='Algorithm',
    show_ci=True,
)
plt.axhline(msve_predict_mean, color='grey', linestyle='--', linewidth=1, alpha=0.7)
plt.title('Variant Comparison at Best Learning Rate (MSVE)')
plt.tight_layout()
plt.savefig(save_path + 'fig3_variant_comparison_msve.png', dpi=150, bbox_inches='tight')
plt.show()

# Same for TD error
plot_learning_curves(
    run_df=combined_run_df,
    config_df=combined_config_df,
    n_bins=200,
    figsize=(10, 5),
    y_col='td_error',
    y_label='TD Error (squared)',
    hue_col='Algorithm',
    legend_title='Algorithm',
    show_ci=True,
)
plt.title('Variant Comparison at Best Learning Rate (TD Error)')
plt.tight_layout()
plt.savefig(save_path + 'fig3_variant_comparison_td_error.png', dpi=150, bbox_inches='tight')
plt.show()

## Figure 4: MSVE Learning Curves by σ (Key Experiment)

σ controls temporal correlation: small σ → consecutive states very similar → warm start informative.

**Prediction**: Streaming iPC's advantage over forward-init should decrease monotonically with σ.

One subplot per σ value, lines per variant, 95% CI from 5 seeds.

In [ ]:
sigma_sweeps = ['td_streaming_sigma', 'td_forward_init_sigma']
sigma_config_dfs = {k: config_dfs[k] for k in sigma_sweeps if k in config_dfs}
sigma_run_dfs = {k: run_dfs[k] for k in sigma_sweeps if k in run_dfs}

if sigma_config_dfs:
    # Best alpha per (variant, sigma)
    best_sigma_config, best_sigma_runs, best_sigma_vals = get_best_ablation_values(
        sigma_config_dfs, sigma_run_dfs,
        sweep_ablation_vars={k: ['ipc.alpha'] for k in sigma_config_dfs},
        sweep_split_vars={k: ['env.sigma'] for k in sigma_config_dfs},
        split_dfs_by_split_vars=True,
        metric_col='msve',
        metric_direction='min',
        metric_type='final_avg',
    )

    combined_sigma_run = pd.concat(list(best_sigma_runs.values()))
    combined_sigma_config = pd.concat(
        [df.assign(split=k) for k, df in best_sigma_config.items()]
    )

    sigma_variant_map = {
        'td_streaming_sigma': 'Streaming iPC',
        'td_forward_init_sigma': 'Forward Init iPC',
    }
    combined_sigma_config['Variant'] = combined_sigma_config['split'].apply(
        lambda x: sigma_variant_map.get(x.split('&')[0], x.split('&')[0])
    )
    combined_sigma_config['σ'] = combined_sigma_config['env.sigma'].astype(str)
    valid_ids = combined_sigma_config['run_id'].unique()
    combined_sigma_run = combined_sigma_run[combined_sigma_run['run_id'].isin(valid_ids)]

    # Fig 4a: MSVE learning curves, one subplot per sigma
    plot_learning_curves(
        run_df=combined_sigma_run,
        config_df=combined_sigma_config,
        n_bins=200,
        figsize=(14, 8),
        y_col='msve',
        y_label='MSVE',
        hue_col='Variant',
        subplot_col='σ',
        subplot_col_label='Step Size σ',
        legend_title='Variant',
        show_ci=True,
    )
    plt.suptitle('MSVE Learning Curves by Step Size σ')
    plt.tight_layout()
    plt.savefig(save_path + 'fig4a_msve_by_sigma.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Sigma sweep data not found')

## Figure 5: Streaming Advantage vs σ (THE Figure)

X-axis: σ. Y-axis: MSVE(forward_init) - MSVE(streaming) at best LR per condition.

A monotonically decreasing curve is strong evidence for the warm-start hypothesis. A flat line would refute it.

In [ ]:
if sigma_config_dfs:
    # Extract asymptotic MSVE per (variant, sigma) at best LR
    # Need per-seed MSVE values for error bars
    advantage_data = []

    for sigma_val in sorted(combined_sigma_config['env.sigma'].unique()):
        for variant_key, variant_label in sigma_variant_map.items():
            mask = (
                (combined_sigma_config['Variant'] == variant_label) &
                (combined_sigma_config['env.sigma'] == sigma_val)
            )
            run_ids = combined_sigma_config.loc[mask, 'run_id'].unique()
            runs = combined_sigma_run[combined_sigma_run['run_id'].isin(run_ids)]

            # Get final MSVE values (last 20% of steps)
            if 'msve' in runs.columns and len(runs) > 0:
                max_step = runs['step'].max()
                tail = runs[runs['step'] >= max_step * 0.8]
                final_msve = tail.groupby('run_id')['msve'].mean()
                for msve_val in final_msve:
                    advantage_data.append({
                        'sigma': sigma_val,
                        'variant': variant_label,
                        'msve': msve_val,
                    })

    adv_df = pd.DataFrame(advantage_data)

    if len(adv_df) > 0:
        # Compute advantage = forward_init - streaming per sigma
        pivot = adv_df.groupby(['sigma', 'variant'])['msve'].agg(['mean', 'std']).reset_index()
        streaming = pivot[pivot['variant'] == 'Streaming iPC'].set_index('sigma')
        forward_init = pivot[pivot['variant'] == 'Forward Init iPC'].set_index('sigma')

        sigmas = sorted(set(streaming.index) & set(forward_init.index))
        advantage_mean = [forward_init.loc[s, 'mean'] - streaming.loc[s, 'mean'] for s in sigmas]
        # Propagated std (approximate)
        advantage_std = [np.sqrt(forward_init.loc[s, 'std']**2 + streaming.loc[s, 'std']**2) for s in sigmas]

        fig, ax = plt.subplots(figsize=(8, 5))
        ax.errorbar(range(len(sigmas)), advantage_mean, yerr=advantage_std,
                     fmt='o-', capsize=5, markersize=8, linewidth=2)
        ax.set_xticks(range(len(sigmas)))
        ax.set_xticklabels([str(s) for s in sigmas])
        ax.set_xlabel('Step Size σ')
        ax.set_ylabel('MSVE(Forward Init) − MSVE(Streaming)')
        ax.axhline(0, color='grey', linestyle='--', linewidth=1, alpha=0.7)
        ax.set_title('Streaming Advantage vs Temporal Correlation')
        plt.tight_layout()
        plt.savefig(save_path + 'fig5_streaming_advantage_vs_sigma.png', dpi=150, bbox_inches='tight')
        plt.show()
else:
    print('Sigma sweep data not found')

## Figure 6: Barrier-Crossing Stratified TD Error

The mechanistic smoking gun. If streaming has lower TD error in open areas but higher error after barrier crossings, that directly confirms the warm-start mechanism.

In [ ]:
# Use the best variant comparison runs (from Fig 3) for barrier-crossing analysis
# These runs have td_error_open and td_error_barrier logged per chunk

if 'td_error_open' in combined_run_df.columns and 'td_error_barrier' in combined_run_df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for ax, metric, title in [
        (axes[0], 'td_error_open', 'Open-Area Transitions'),
        (axes[1], 'td_error_barrier', 'Barrier-Crossing Transitions'),
    ]:
        plot_learning_curves(
            run_df=combined_run_df,
            config_df=combined_config_df,
            n_bins=200,
            y_col=metric,
            y_label='TD Error (squared)',
            hue_col='Algorithm',
            legend_title='Algorithm',
            show_ci=True,
            ax=ax,
        )
        ax.set_title(title)

    plt.suptitle('TD Error Stratified by Barrier Crossing')
    plt.tight_layout()
    plt.savefig(save_path + 'fig6_barrier_stratified_td_error.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    # Fallback: plot from individual sweeps if available
    print('Barrier-stratified metrics not found in combined data. Check that td_error_open and td_error_barrier are logged.')

## Figure 7: T Sensitivity

Does T=1 still work when targets change every step? If streaming at T=1 matches forward-init at higher T, persistent nodes substitute for inference iterations.

In [ ]:
if 'td_streaming_T' in config_dfs:
    T_config = config_dfs['td_streaming_T']
    T_runs = run_dfs['td_streaming_T']

    # Best alpha per T
    best_T_config, best_T_runs, best_T_vals = get_best_ablation_values(
        {'td_streaming_T': T_config},
        {'td_streaming_T': T_runs},
        sweep_ablation_vars={'td_streaming_T': ['ipc.alpha']},
        sweep_split_vars={'td_streaming_T': ['ipc.T']},
        split_dfs_by_split_vars=True,
        metric_col='msve',
        metric_direction='min',
        metric_type='final_avg',
    )

    combined_T_run = pd.concat(list(best_T_runs.values()))
    combined_T_config = pd.concat(
        [df.assign(split=k) for k, df in best_T_config.items()]
    )
    combined_T_config['T'] = combined_T_config['ipc.T'].astype(str)
    valid_ids = combined_T_config['run_id'].unique()
    combined_T_run = combined_T_run[combined_T_run['run_id'].isin(valid_ids)]

    # Fig 7a: MSVE learning curves by T
    plot_learning_curves(
        run_df=combined_T_run,
        config_df=combined_T_config,
        n_bins=200,
        figsize=(10, 5),
        y_col='msve',
        y_label='MSVE',
        hue_col='T',
        legend_title='T (inference steps)',
        show_ci=True,
    )
    plt.axhline(msve_predict_mean, color='grey', linestyle='--', linewidth=1, alpha=0.7)
    plt.title('Streaming iPC: Effect of T on MSVE')
    plt.tight_layout()
    plt.savefig(save_path + 'fig7a_T_sensitivity_msve.png', dpi=150, bbox_inches='tight')
    plt.show()

    # T sensitivity bar plot
    plot_param_sensitivity(
        run_df=T_runs,
        config_df=T_config,
        x_col='ipc.T',
        title='Streaming iPC: T Sensitivity',
        x_label='Inference Steps (T)',
        y_label='MSVE',
        metric_col='msve',
        metric_type='final_avg',
        hue_col='ipc.alpha',
        legend_title='Alpha',
        pow_2_legend=True,
    )
    plt.savefig(save_path + 'fig7b_T_sensitivity.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('T sweep data not found')

## Summary

TODO: Fill in after sweeps are complete.

| Method | Best MSVE | Best LR | Notes |
|--------|-----------|---------|-------|
| Streaming iPC | | | |
| Forward-init iPC | | | |
| BP (Adam) | | | |
| Linear BP | | | |
| Predict mean | | N/A | Baseline |